# Example queries: `timeseries` (comstock_oedi)

Auto-generated from `tests/query_snapshots/timeseries.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [1]:
from pathlib import Path
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [2]:
# This notebook lives in `tests/example_notebooks/`; the snapshot test
# cache is its sibling `tests/query_snapshots/comstock_oedi_cache/`. Resolve
# the path relative to the notebook directory (`_dh[0]` is set by
# IPython at kernel startup; falls back to CWD outside Jupyter).
_NB_DIR = Path(_dh[0] if "_dh" in globals() else ".").resolve()
_CACHE = (_NB_DIR / "../query_snapshots/comstock_oedi_cache").resolve()
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "comstock_amy2018_r2_2025",
    buildstock_type="comstock",
    db_schema="comstock_oedi_state_and_county",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


INFO:buildstock_query.query_core:Loading comstock_amy2018_r2_2025 ...


INFO:botocore.tokens:Loading cached SSO token for nrel-sso


## `ts_monthly_electricity_by_state`

Monthly electricity grouped by state (CO only), upgrade 0 baseline.


In [3]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    timestamp_grouping_func='month',
    group_by=['state', 'time'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,state,timestamp,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,CO,2018-01-01,134649,84415.71204,2976,1.385482e+09
1,CO,2018-02-01,134649,84415.71204,2688,1.280966e+09
2,CO,2018-03-01,134649,84415.71204,2976,1.237644e+09
3,CO,2018-04-01,134649,84415.71204,2880,1.171385e+09
4,CO,2018-05-01,134649,84415.71204,2976,1.289014e+09


## `ts_monthly_two_fuels_by_building_type`

Monthly electricity + natural gas grouped by building type and time, CO only.


In [4]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption', 'out.natural_gas.total.energy_consumption'],
    annual_only=False,
    timestamp_grouping_func='month',
    group_by=['comstock_building_type', 'time'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,comstock_building_type,timestamp,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption,natural_gas.total.energy_consumption
0,FullServiceRestaurant,2018-01-01,13461,4556.431396,2976,1.042121e+08,1.283346e+08
1,FullServiceRestaurant,2018-02-01,13461,4556.431396,2688,9.695190e+07,1.206407e+08
2,FullServiceRestaurant,2018-03-01,13461,4556.431396,2976,9.693666e+07,1.067041e+08
3,FullServiceRestaurant,2018-04-01,13461,4556.431396,2880,9.240466e+07,9.565136e+07
4,FullServiceRestaurant,2018-05-01,13461,4556.431396,2976,9.842727e+07,8.398370e+07


## `ts_year_collapse_electricity`

Year-collapsed timeseries electricity, grouped by building type, CO only.


In [5]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    timestamp_grouping_func='year',
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,comstock_building_type,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,FullServiceRestaurant,13461,4556.431396,35040,1.209201e+09
1,Hospital,73,152.003100,35040,1.369473e+09
2,LargeHotel,2815,3317.336942,35040,1.128185e+09
3,LargeOffice,2271,849.927533,35040,1.546956e+09
4,MediumOffice,8708,2418.404703,35040,1.298819e+09


## `ts_15min_raw_electricity_by_state`

Raw 15-min cadence timeseries (no timestamp_grouping_func), CO only. Exercises the un-aggregated time-axis code path that example notebooks rely on.


In [6]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    group_by=['state', 'time'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,state,timestamp,sample_count,units_count,electricity.total.energy_consumption
0,CO,2018-09-12 00:45:00,134649,84415.71204,321435.852724
1,CO,2018-09-12 01:00:00,134649,84415.71204,317466.909737
2,CO,2018-09-12 01:15:00,134649,84415.71204,303706.205138
3,CO,2018-09-12 01:30:00,134649,84415.71204,300550.875612
4,CO,2018-09-12 01:45:00,134649,84415.71204,297504.646372


## `ts_daily_electricity_by_state`

Daily-grouped timeseries electricity, CO only. Exercises date_trunc('day', ...) plus the rows_per_sample / distinct-bs-keys grouping_metrics branch.


In [7]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    timestamp_grouping_func='day',
    group_by=['state', 'time'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,state,timestamp,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,CO,2018-01-01,134649,84415.71204,96,5.631904e+07
1,CO,2018-01-02,134649,84415.71204,96,5.220220e+07
2,CO,2018-01-03,134649,84415.71204,96,4.810277e+07
3,CO,2018-01-04,134649,84415.71204,96,4.638417e+07
4,CO,2018-01-05,134649,84415.71204,96,4.562267e+07


## `ts_hourly_electricity_by_building_type`

Hourly-grouped timeseries electricity, CO + single building type. Exercises date_trunc('hour', ...) — the smallest grouping_func granularity.


In [8]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    timestamp_grouping_func='hour',
    group_by=['comstock_building_type', 'time'],
    restrict=[('state', ['CO']), ('comstock_building_type', ['Warehouse'])],
)
result.head() if hasattr(result, 'head') else result


,comstock_building_type,timestamp,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,Warehouse,2018-01-01 00:00:00,18058,21505.528073,4,425794.630349
1,Warehouse,2018-01-01 01:00:00,18058,21505.528073,4,430400.931127
2,Warehouse,2018-01-01 02:00:00,18058,21505.528073,4,364028.417190
3,Warehouse,2018-01-01 03:00:00,18058,21505.528073,4,371733.766567
4,Warehouse,2018-01-01 04:00:00,18058,21505.528073,4,375253.339734


## `ts_monthly_multi_attribute_group_by`

Monthly TS electricity grouped by [building_type, state, time] — multi-attribute group_by combined with the time axis. Every other TS entry uses single-attribute + time; this pins the SQL shape for the more general case.


In [9]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    timestamp_grouping_func='month',
    group_by=['comstock_building_type', 'state', 'time'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,comstock_building_type,state,timestamp,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,FullServiceRestaurant,CO,2018-01-01,13461,4556.431396,2976,1.042121e+08
1,FullServiceRestaurant,CO,2018-02-01,13461,4556.431396,2688,9.695190e+07
2,FullServiceRestaurant,CO,2018-03-01,13461,4556.431396,2976,9.693666e+07
3,FullServiceRestaurant,CO,2018-04-01,13461,4556.431396,2880,9.240466e+07
4,FullServiceRestaurant,CO,2018-05-01,13461,4556.431396,2976,9.842727e+07


## `ts_hourly_electricity_by_state`

Hourly TS electricity grouped by state, CO only. Restrict and group_by deliberately match ts_daily_electricity_by_state and ts_monthly_electricity_by_state so the hourly→daily→monthly sum invariants align.


In [10]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    timestamp_grouping_func='hour',
    group_by=['state', 'time'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,state,timestamp,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,CO,2018-01-01 00:00:00,134649,84415.71204,4,2.216948e+06
1,CO,2018-01-01 01:00:00,134649,84415.71204,4,2.177930e+06
2,CO,2018-01-01 02:00:00,134649,84415.71204,4,1.874028e+06
3,CO,2018-01-01 03:00:00,134649,84415.71204,4,1.832707e+06
4,CO,2018-01-01 04:00:00,134649,84415.71204,4,1.820178e+06


## `ts_year_by_county_co`

TS year-collapse grouped by county, CO only. Comstock-only because comstock's metadata is denormalized at (bldg_id, tract, state) granularity and a building's tracts can map to different counties; resstock's md is one row per bldg, so the dis-aggregation behavior under test is structurally absent. Pins the bs_per_bldg shape that carries bs-side group_by columns as TRUE GROUP BY keys (not arbitrary()) so per-tract weight slices land in the right county. Sum across counties must equal the no-county-group total — see test_ts_year_county_disaggregation in test_invariants.py.


In [11]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    timestamp_grouping_func='year',
    group_by=['in.county_name'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,county_name,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,"CO, Adams County",10170,8130.650795,35040,1.542441e+09
1,"CO, Alamosa County",447,231.643122,35040,2.512239e+07
2,"CO, Arapahoe County",13494,8111.235765,35040,1.740775e+09
3,"CO, Archuleta County",324,152.825688,35040,1.323592e+07
4,"CO, Baca County",78,32.398055,35040,4.982582e+06


## `ts_year_by_tract_co`

TS year-collapse grouped by tract (in.nhgis_tract_gisjoin), CO only. Comstock-only — exercises the finest-grained bs-side group_by, where bs_per_bldg's GROUP BY (bldg, state, tract) restores 1:1 row-to-tract mapping. Cross-checks: sum across tracts must equal the by-county total (a tract belongs to exactly one county) and the no-group total. Smallest-meaningful unit at which the tract fan-out manifests; if bs_per_bldg collapses tract via arbitrary() we'd see 1 row per (bldg, state) instead of 1 per metadata row.


In [12]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    timestamp_grouping_func='year',
    group_by=['in.nhgis_tract_gisjoin'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,nhgis_tract_gisjoin,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,G0800010007801,148,109.689608,35040,8.881154e+06
1,G0800010007802,115,42.020192,35040,8.831946e+06
2,G0800010007900,104,34.658314,35040,6.320033e+06
3,G0800010008000,46,14.126677,35040,4.262248e+06
4,G0800010008100,37,32.023338,35040,1.274415e+07


## `ts_year_overall_co`

TS year-collapse with NO bs-side group_by, CO only. Comstock-only sibling to the by-county / by-tract variants; serves as the cross-check baseline (sum-across-county and sum-across-tract must equal this value within rtol=1e-3). Without this anchor, a bug that silently dropped tract-fractional weight would still produce by-county and by-tract totals that agreed WITH EACH OTHER (both equally wrong) — only against the no-group total can we distinguish the right answer.


In [13]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    timestamp_grouping_func='year',
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,134649,84415.71204,35040,1.600424e+10
